### visual representation

- "视觉token和文本token其实是很不同的，文本token就是一个几十万水平的整数，经过embedding之后也仍然只是几十万独立向量值。但该方案的视觉token并没有经过离散化过程，而是直接投影到1280维内部表示的，相当于是个1280维的embedding。"
    - 文本 token 的确来自一个有限词表的离散 ID，查表后变成连续嵌入；很多多模态方案里的“视觉 token”并非离散化，而是图像 patch 经过编码/投影得到的连续向量序列（维度可能是 768/1024/1280 等）。因此统计这些视觉嵌入的数值，分布会呈现近似连续、取值非常丰富——但别忘了，文本嵌入向量本身也是连续的，只是它们来自一个有限个候选向量的查表集合。
    - 文本有 fixed 的 vocabulary，通过 lookup table 获得 embedding（查表嵌入），visual 没有码本索引；
    - 文本可以 decode 生成，visual embedding 往往只是做 project 和对齐（只作为输入），不直接 decode visual 
    - https://mp.weixin.qq.com/s/hW22w07Ro4-xCOs7SwljiA
- 1 visual embedding >> 1 text token embedding
    - 高密度信息Token

### architecture

- encoder
    - SAM => Conv Compressor => CLIP
        - SAM: 切 patch，代表着局部细节的特征令牌。例如，对于一张1024x1024的图片，会产生4096个图块令牌。
        - Convolutional Compressor: 这个模块像一个信息漏斗，将输入的4096个令牌通过卷积操作进行下采样，数量锐减16倍，变成仅仅256个令牌。
        - CLIP: CLIP接收这256个浓缩后的令牌，对它们进行全局的关联和理解，将图像的整体内容（比如布局、语义）和丰富的概念知识编码进来。
- training
    - 阶段一：独立训练DeepEncoder (3.5.1 Training DeepEncoder)，预训练为起点；
    - 阶段二：端到端训练完整的DeepSeek-OCR模型 (3.5.2 Training DeepSeek-OCR)

### compression

- when the number of text tokens is within 10 times that of vision tokens (i.e., a compression ratio < 10×), the model can achieve decoding (OCR) precision of 97%.
    - 结果见 table 7
    - “97%的解码精度”意味着，模型解码生成的文本内容与图片原始的、正确的文本内容（即“ground truth”，标准答案）相比，其准确率达到了97%。

### prompt template

- `<image>\n<|grounding|>Convert the document to markdown.`
- `<image>\nParse the figure.`
- `<image>\nDescribe this image in detail.`

### rendering

- 除了 ocr 还有 det 能力（甚至说每一个子文本（空间中连续）、子图区域都会对应一个 `<|det|><|/det|>`）
    - 所有能扣出原图中的一些子图，当然也包括原始文本的 det 区域，然后进行精准的 rendering
    - tag
        - `<|ref|>title<|/ref|>`
            - `<|ref|>sub_title<|/ref|>`
        - `<|ref|>text<|/ref|>`
        - `<|ref|>image<|/ref|>`
        - `<--- Page Split --->`: 分页（比如对于 pdf）